# RoofTop Regression Scoping

## Purpose

Scope the APTs (Address Points) that were **relocated for ESP** so they can be validated
and, once relocation is confirmed, fed into the Data Correctness (relocation) process.

### Background (SEACO-6171)

In Q1–Q2 2026, a total of **8,569,598 (~8.5M)** APTs were ingested across multiple sources
(e.g. `CATASTRO_APT`), with locations updated according to each respective source. Some of
these ingestions moved existing APTs (originally from Genesis, Nomecalles Opensource,
APT Eustat, esp event_lost) away from their rooftop, causing a regression in the ESP RAC
(Rooftop Accuracy) metrics.

### Scope of this notebook

As the next step in the investigation, this notebook:

1. Determines the exact count of APTs that moved **outside the Building Footprint (BFP)**
   in Q1 and Q2 for ESP.
2. Scopes those relocated APTs.
3. Prepares the scoped set for validation, after which the Data Correctness process for
   relocation can be triggered.

Reference: [SEACO-6171](https://tomtom.atlassian.net/browse/SEACO-6171?focusedCommentId=12144061)

## Approach

The relocated-APT scoping is built step by step from two snapshots (OLD vs LATEST) and the
Building Footprint (BFP) layer. The BFP intersection is computed **per snapshot** (on OLD and
on LATEST independently) and only then are the two sides joined:

1. **Load the OLD snapshot** into a Delta table (pin the older revision to compare against).
2. **Load the LATEST snapshot** into a Delta table (resolve the newest revision).
3. **Read the OLD snapshot** Delta table as a `Dataset[OrbisElement]`.
4. **Read the LATEST snapshot** Delta table as a `Dataset[OrbisElement]`.
5. **Filter both snapshots by `countryISO` = `ESP`** (value of the `metadata:country` tag).
6. **Filter the OLD snapshot by `metadata:updated`** — keep only APTs whose `metadata:updated`
   value (an epoch timestamp) is **greater than 1st Feb 2026** (the regression window start).
7. **Load the Building Footprint (BFP) snapshot** (MCR `preprocess_prod.bfp`, filtered to the
   ESP license zone) and build its geometry.
8. **Enrich each snapshot with its BFP match.** For BOTH the OLD and the LATEST dataset,
   spatially join the APT point against the BFP polygons (`ST_Intersects`) and add, per APT:
   - `is_inside_bfp` — `true` when the APT point falls inside a building footprint;
   - `bfp_id` and `bfp_wkt` — the id and geometry (WKT) of the matched footprint (null when no match).
9. **Left join** the enriched OLD snapshot with the enriched LATEST snapshot on the APT id
   (keep all OLD rows; LATEST columns are null where the APT is missing in the latest).
10. **Add geometry / distance columns** on the joined dataset:
    - `old_wkt` and `latest_wkt` — `POINT(lng lat)` built from each snapshot's lat/lng;
    - `distance_in_meters` — great-circle (Haversine) distance between the OLD and LATEST point;
    - `hookline` — `LINESTRING(old -> latest)` connecting the two points.
11. **Final dataset** contains, per APT:
    - OLD snapshot details (location, tags, `is_inside_bfp`, `bfp_id`, `bfp_wkt`),
    - LATEST snapshot details (location, tags, `is_inside_bfp`, `bfp_id`, `bfp_wkt`),
    - `distance_in_meters` and `hookline`.

In [0]:
%sql
-- Drop the scoping databases if they already exist, then recreate them fresh.
-- CASCADE removes any tables left from a previous run. Run this once before the loader cells.
DROP DATABASE IF EXISTS rooftop_regression_scoping_db_old CASCADE;
DROP DATABASE IF EXISTS rooftop_regression_scoping_db_new CASCADE;
CREATE DATABASE IF NOT EXISTS rooftop_regression_scoping_db_old;
CREATE DATABASE IF NOT EXISTS rooftop_regression_scoping_db_new;

In [0]:
%scala
// Shared constants used across the cells below (defined once here; values do not change between runs).
val LAYER_ID = 14533
val COUNTRY_ISO = "ESP"
val OLD_DATABASE = "rooftop_regression_scoping_db_old"
val NEW_DATABASE = "rooftop_regression_scoping_db_new"
val oldSnapshotTable = s"${OLD_DATABASE}.apt_snapshot_old"
val latestSnapshotTable = s"${NEW_DATABASE}.apt_snapshot_new"

In [0]:
%scala
// Step 1 - Load the OLD snapshot into a Delta table (filtered to country = ESP).
// Pin OLD_REVISION to the older revision you want to compare the latest snapshot against.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

// >>> Set the OLD snapshot revision id here <<<
val OLD_REVISION = 42394153L

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

println(s"Loading OLD snapshot revision: $OLD_REVISION")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, OLD_REVISION)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(OLD_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded old snapshot, keeping only APTs whose metadata:country tag = ESP
val oldAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
oldAptDS.write.format("delta").mode("overwrite").saveAsTable(oldSnapshotTable)
println(s"OLD snapshot written to: $oldSnapshotTable")
// display(oldAptDS)

In [0]:
%scala
// Step 2 - Load the LATEST snapshot into a Delta table (filtered to country = ESP).
// The newest revision is resolved automatically via LoadSnapshotInfo.getLatestRevisionId.

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(LAYER_ID)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(LAYER_ID, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(NEW_DATABASE)
new LoadFreshSnapshotData().run()

// Persist the loaded latest snapshot, keeping only APTs whose metadata:country tag = ESP
val latestAptDS = new OrbisElementRepository(LAYER_ID.toString).readAll
  .filter(exists(col("tags"), t =>
    t.getField("tagKey").getField("key") === "metadata:country"
    && t.getField("value") === COUNTRY_ISO))
latestAptDS.write.format("delta").mode("overwrite").saveAsTable(latestSnapshotTable)
println(s"LATEST snapshot written to: $latestSnapshotTable")
// display(latestAptDS)

In [0]:
%scala
// Step 2 - Load the LATEST snapshot into a Delta table 

import com.databricks.dbutils_v1.DBUtilsHolder
import com.fasterxml.jackson.databind.{DeserializationFeature, ObjectMapper}
import com.tomtom.addressing.bulk.commons.model.LayerVersions
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadFreshSnapshotData
import com.tomtom.orbis.addressing.bulk.commons.odp.LoadSnapshotInfo
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader
import com.tomtom.orbis.addressing.bulk.commons.repository.OrbisElementRepository
import org.apache.spark.sql.functions._

val mapper = new ObjectMapper()
  .configure(DeserializationFeature.FAIL_ON_UNKNOWN_PROPERTIES, false)
  .configure(DeserializationFeature.ACCEPT_EMPTY_STRING_AS_NULL_OBJECT, true)
  .configure(DeserializationFeature.ACCEPT_SINGLE_VALUE_AS_ARRAY, true)

val env = "PROD"
implicit val sparky = spark
ConfigLoader.forEnvironment(env.toLowerCase)

val latestRevision = LoadSnapshotInfo.getLatestRevisionId(6)
println(s"Loading LATEST snapshot revision: ${latestRevision.get}")
val versionsBuilder = LayerVersions.builder()
versionsBuilder.layer(6, latestRevision.get)
val versionMetadata: String = mapper.writeValueAsString(versionsBuilder.build())
DBUtilsHolder.dbutils.widgets.text("layer-versions", versionMetadata)

SparkHelper.init(NEW_DATABASE)
new LoadFreshSnapshotData().run()

In [0]:
%scala
// Step 3 - Read the OLD snapshot Delta table as a Dataset[OrbisElement].
// Enrich each row with country, the unsigned colon-separated orbisIdString, and metadata:updated (epoch).

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.{Id, OrbisElement}
import java.lang.{Long => JLong}

// Build the colon-separated unsigned id string: {layerId}:{unsignedHigh}:{unsignedLow}
def convertOrbisIdToString(orbisId: Id): String = {
  val COLON_SEPARATOR = ":"
  Seq(orbisId.layerId.getOrElse(14533).toString,
      JLong.toUnsignedString(orbisId.high),
      JLong.toUnsignedString(orbisId.low)).mkString(COLON_SEPARATOR)
}
val convertOrbisIdUDF = udf((orbisId: Id) => convertOrbisIdToString(orbisId))

val countryTagKey = "metadata:country"
val updatedTagKey = "metadata:updated"

val oldSnapshotDS: Dataset[OrbisElement] =
  spark.table(oldSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val oldSnapshotEnriched = oldSnapshotDS
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))
  // metadata:updated value is an epoch timestamp string -> cast to long for the date filter in Step 6
  .withColumn("updated_epoch",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === updatedTagKey)
      .getItem(0).getField("value").cast("long"))

println(s"OLD snapshot read from: $oldSnapshotTable, count: ${oldSnapshotEnriched.count()}")
//display(oldSnapshotEnriched)

In [0]:
%scala
// Step 4 - Read the LATEST snapshot Delta table as a Dataset[OrbisElement].
// Enrich each row with country and the unsigned colon-separated orbisIdString (UDF defined in Step 3).

import org.apache.spark.sql.{Dataset, Encoders}
import org.apache.spark.sql.functions._
import com.tomtom.orbis.io.spark.model.OrbisElement

val latestSnapshotDS: Dataset[OrbisElement] =
  spark.table(latestSnapshotTable).as[OrbisElement](Encoders.product[OrbisElement])

val latestSnapshotEnriched = latestSnapshotDS
  .withColumn("country",
    filter(col("tags"), t => t.getField("tagKey").getField("key") === countryTagKey)
      .getItem(0).getField("value"))
  .withColumn("orbisIdString", convertOrbisIdUDF(col("id")))

println(s"LATEST snapshot read from: $latestSnapshotTable, count: ${latestSnapshotEnriched.count()}")
// display(latestSnapshotEnriched)

In [0]:
%scala
// Step 5 - Filter both snapshots to country ISO = ESP.
// (Snapshots are already loaded ESP-only in Steps 1-2; this re-applies the filter defensively.)

import org.apache.spark.sql.functions._

val oldEsp = oldSnapshotEnriched.filter(col("country") === COUNTRY_ISO)
val latestEsp = latestSnapshotEnriched.filter(col("country") === COUNTRY_ISO)

println(s"OLD ESP count: ${oldEsp.count()}")
println(s"LATEST ESP count: ${latestEsp.count()}")

In [0]:
%scala
// Step 6 - Filter the OLD snapshot to APTs updated after 1st Feb 2026 (the regression window start).
// metadata:updated is an epoch timestamp in SECONDS; adjust the constant if your values are in millis.

import org.apache.spark.sql.functions._

// 2026-02-01T00:00:00Z in epoch SECONDS. Use 1769904000000L if metadata:updated is in milliseconds.
val FEB_FIRST_EPOCH = 1769904000L

val oldEspUpdated = oldEsp.filter(col("updated_epoch") > FEB_FIRST_EPOCH)

println(s"OLD ESP updated after 1st Feb count: ${oldEspUpdated.count()}")
// display(oldEspUpdated)

In [0]:
%scala
// Step 7 (a) - Refresh the MCR Building Footprint delta table via LoadDataFromMcr.
// The loader writes BFP to <deltaConfig database>.bfp; in PROD that is preprocess_prod.bfp (read next cell).

/*
import org.apache.spark.sql.{Dataset, SparkSession}
import com.tomtom.orbis.addressing.bulk.commons.spark.SparkHelper
import com.tomtom.addressing.bulk.commons.config.ConfigLoader
import com.tomtom.addressing.bulk.scala.load.LoadDataFromMcr
import com.tomtom.addressing.bulk.commons.config.SourceConfigLoader

val env = "PROD" // dbutils.widgets.get("environment").toUpperCase()
println(s"Environment Name $env")

val config = SourceConfigLoader.getConfig

// deltaConfig resolves to the source_data table for the env; its database (preprocess_prod in PROD)
// is where LoadDataFromMcr persists the BFP table -> preprocess_prod.bfp
val deltaConfig = config.getTableMapping.getOrDefault(env, "preprocess_dev.source_data")
val Array(bfpDatabaseName, tableName) = deltaConfig.split("\\.")
println(s"BFP database: $bfpDatabaseName")

implicit val sparky = spark
implicit val envConfig = ConfigLoader.forEnvironment(env.toLowerCase)
SparkHelper.init(bfpDatabaseName)

new LoadDataFromMcr().run()
*/

In [ ]:
%scala
// Read the loaded road-element layer (layer 6) from the NEW database, keep only rows that
// have a geometry (wkt is not null), and display them.

import org.apache.spark.sql.functions._

val roadElementDataset = spark.table(s"${NEW_DATABASE}.layer_6")
  .filter(col("wkt").isNotNull)

println(s"Road element (layer_6) rows with wkt: ${roadElementDataset.count()}")
display(roadElementDataset)

In [0]:
%scala
// Step 7 (b) - Read the ESP Building Footprint (BFP) polygons from preprocess_prod.bfp.
// Filter to the ESP license zone and build the polygon geometry for the spatial join.

import org.apache.spark.sql.functions._
import org.apache.sedona.spark.SedonaContext

// Sedona is required for the ST_Intersects spatial join in the next cell
val sedona = SedonaContext.create(spark)

val espBfpDataset = spark.table("preprocess_prod.bfp")
  .filter(col("licenseZone") === "ESP")
  .filter(col("wkt").isNotNull)
  .withColumn("bfp_geom", expr("ST_GeomFromWKT(wkt)"))

println(s"ESP BFP polygon count: ${espBfpDataset.count()}")
display(espBfpDataset)

In [0]:
%scala
// Step 8 - Enrich BOTH snapshots with their BFP match (per snapshot, before the join).
// An INNER ST_Intersects join (Sedona plans a partitioned RangeJoin, no broadcast) finds the
// matched footprint; a cheap equi LEFT join back onto the full APT set then yields
// is_inside_bfp, bfp_id and bfp_wkt without dropping APTs that fall outside every footprint.

import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions._

// A spatial LEFT join is NOT optimized by Sedona -> Spark uses a BroadcastNestedLoopJoin and
// broadcasts the whole ~6 GB BFP relation (EXECUTOR_BROADCAST_JOIN_OOM). Disable the broadcast
// threshold so the inner spatial join below is planned as a Sedona partitioned RangeJoin.
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

// BFP polygons prepared once for the spatial join
val bfpForJoin = espBfpDataset
  .select(col("bfpId").as("bfp_id"), col("wkt").as("bfp_wkt"), col("bfp_geom"))

// For one snapshot: find the (single) intersecting footprint per APT via an inner spatial join,
// then left-join it back onto every APT so non-matching APTs are kept with is_inside_bfp = false.
def enrichWithBfp(apts: DataFrame): DataFrame = {
  val aptCols = apts.select(col("orbisIdString"), col("lat"), col("lng"), col("tags"))

  val aptPoints = aptCols
    .withColumn("apt_geom", expr("ST_GeomFromText(concat('POINT(', lng, ' ', lat, ')'))"))

  // INNER spatial join -> Sedona RangeJoin (no broadcast). Keep one footprint per APT.
  val matched = aptPoints.alias("apt")
    .join(bfpForJoin.alias("bfp"), expr("ST_Intersects(bfp.bfp_geom, apt.apt_geom)"))
    .select(
      col("apt.orbisIdString").as("orbisIdString"),
      col("bfp.bfp_id").as("bfp_id"),
      col("bfp.bfp_wkt").as("bfp_wkt"))
    .dropDuplicates("orbisIdString")

  // Equi LEFT join back onto the full APT set (no spatial predicate -> cheap shuffle join)
  aptCols
    .join(matched, Seq("orbisIdString"), "left")
    .withColumn("is_inside_bfp", col("bfp_id").isNotNull)
}

val oldEnriched = enrichWithBfp(oldEspUpdated)
val latestEnriched = enrichWithBfp(latestEsp)

println(s"OLD enriched: ${oldEnriched.count()}, inside BFP: ${oldEnriched.filter(col("is_inside_bfp")).count()}")
println(s"LATEST enriched: ${latestEnriched.count()}, inside BFP: ${latestEnriched.filter(col("is_inside_bfp")).count()}")
//display(oldEnriched)

In [ ]:
%scala
// Steps 9-10 - Left join enriched OLD with enriched LATEST on orbisIdString, then add geometry/distance:
// old_wkt / latest_wkt (POINT(lng lat)), distance_in_meters (Haversine) and hookline (LINESTRING old->latest).

import org.apache.spark.sql.functions._

val oldRenamed = oldEnriched
  .withColumn("old_wkt", concat(lit("POINT("), col("lng"), lit(" "), col("lat"), lit(")")))
  .withColumnRenamed("lat", "old_lat")
  .withColumnRenamed("lng", "old_lon")
  .withColumnRenamed("tags", "old_tags")
  .withColumnRenamed("is_inside_bfp", "old_is_inside_bfp")
  .withColumnRenamed("bfp_id", "old_bfp_id")
  .withColumnRenamed("bfp_wkt", "old_bfp_wkt")

val latestRenamed = latestEnriched
  .withColumn("latest_wkt", concat(lit("POINT("), col("lng"), lit(" "), col("lat"), lit(")")))
  .withColumnRenamed("lat", "latest_lat")
  .withColumnRenamed("lng", "latest_lon")
  .withColumnRenamed("tags", "latest_tags")
  .withColumnRenamed("is_inside_bfp", "latest_is_inside_bfp")
  .withColumnRenamed("bfp_id", "latest_bfp_id")
  .withColumnRenamed("bfp_wkt", "latest_bfp_wkt")

// Haversine distance in meters (mean earth radius) - avoids Sedona for the distance calc
val EARTH_RADIUS_M = 6371008.8
val dLat = radians(col("latest_lat") - col("old_lat"))
val dLon = radians(col("latest_lon") - col("old_lon"))
val haversine =
  pow(sin(dLat / 2), 2) +
  cos(radians(col("old_lat"))) * cos(radians(col("latest_lat"))) * pow(sin(dLon / 2), 2)
val distanceMeters = lit(2 * EARTH_RADIUS_M) * asin(sqrt(haversine))

val oldVsLatest = oldRenamed
  .join(latestRenamed, Seq("orbisIdString"), "left")
  .withColumn("distance_in_meters",
    when(col("old_wkt").isNotNull && col("latest_wkt").isNotNull, distanceMeters))
  .withColumn("hookline",
    when(col("old_wkt").isNotNull && col("latest_wkt").isNotNull,
      concat(lit("LINESTRING("),
        col("old_lon"), lit(" "), col("old_lat"), lit(", "),
        col("latest_lon"), lit(" "), col("latest_lat"), lit(")"))))

println(s"oldVsLatest rows where distance_in_meters != 0: ${oldVsLatest.filter(col("distance_in_meters") =!= 0).count()}")

In [ ]:
%scala
// Step 11 - Assemble, filter and write the final scoped relocated-APT dataset as single CSV files.
// Keep only the regression candidates: APT was inside a BFP in OLD, is outside in LATEST,
// and the latest edit was NOT made by VERTEX (latest_tags metadata:user != VERTEX).
// Three CSVs are written: one WITH the tag columns, one WITHOUT, and the DC relocation input format.

import org.apache.spark.sql.DataFrame
import org.apache.spark.sql.functions._

// Never auto-broadcast the large spatial relations (BFP / road layer) — keep this set here too so
// the joins below don't depend on an earlier cell having run (avoids EXECUTOR_BROADCAST_JOIN_OOM).
spark.conf.set("spark.sql.autoBroadcastJoinThreshold", -1)

val finalScopedBase = oldVsLatest
  .select(
    col("orbisIdString"),
    col("old_lat"), col("old_lon"), col("old_wkt"), col("old_tags"),
    col("old_is_inside_bfp"), col("old_bfp_id"), col("old_bfp_wkt"),
    col("latest_lat"), col("latest_lon"), col("latest_wkt"), col("latest_tags"),
    col("latest_is_inside_bfp"), col("latest_bfp_id"), col("latest_bfp_wkt"),
    col("distance_in_meters"),
    col("hookline"))
  // Regression scope: inside BFP before, outside BFP after, and the latest edit
  // was NOT made by VERTEX (latest_tags has no metadata:user = VERTEX)
  .filter(col("old_is_inside_bfp") === true
    && col("latest_is_inside_bfp") === false
    && !exists(col("latest_tags"), t =>
      t.getField("tagKey").getField("key") === "metadata:user"
      && t.getField("value") === "VERTEX"))
  // Flag whether the APT carries metadata:apa:improvement = yes in old_tags
  .withColumn("is_APA_Improvement_exist",
    exists(col("old_tags"), t =>
      t.getField("tagKey").getField("key") === "metadata:apa:improvement"
      && t.getField("value") === "yes"))
  // The scope is reused by both the road-intersection join and the downstream writes, so cache it.
  .cache()

// Flag whether the APT's hookline (old -> latest line) intersects any road element (layer 6) and,
// when it does, capture the colon-separated road id(s) it crosses (via convertOrbisIdUDF from Step 3).
// The scoped hookline set is tiny vs the ~6 GB road layer, so BROADCAST the small (hookline) side:
// this makes Sedona plan a BroadcastIndexJoin (spatial index built on the broadcast side) and
// guarantees the large road relation is never broadcast (the cause of EXECUTOR_BROADCAST_JOIN_OOM).
// A hookline can cross several roads, so the matched road ids are collected into one comma-separated
// string per APT; a LEFT join back onto the full scope then yields the flag (false where none cross).
val roadGeomForJoin = roadElementDataset
  .withColumn("road_geom", expr("ST_GeomFromWKT(wkt)"))
  .withColumn("road_id_string", convertOrbisIdUDF(col("id")))
  .select(col("road_id_string"), col("road_geom"))

val scopedHooklines = finalScopedBase
  .filter(col("hookline").isNotNull)
  .select(col("orbisIdString"), expr("ST_GeomFromText(hookline)").as("hookline_geom"))

val roadIntersected = roadGeomForJoin.alias("road")
  .join(broadcast(scopedHooklines).alias("apt"),
    expr("ST_Intersects(road.road_geom, apt.hookline_geom)"))
  .select(col("apt.orbisIdString").as("orbisIdString"), col("road.road_id_string").as("road_id_string"))
  .groupBy("orbisIdString")
  .agg(concat_ws(",", collect_set(col("road_id_string"))).as("intersecting_road_element_ids"))
  .withColumn("is_intersects_with_road_element", lit(true))

val finalScoped = finalScopedBase
  .join(roadIntersected, Seq("orbisIdString"), "left")
  .withColumn("is_intersects_with_road_element",
    coalesce(col("is_intersects_with_road_element"), lit(false)))

val outputDir = "/mnt/aqua/data-correctness/correction-input/SEACO-6182-ESP-Relocation-Scope"

// Write a DataFrame as one named .csv file: Spark emits a folder of part files, so coalesce to a
// single partition, then promote that part file to the target name and clean up the temp dir.
def writeSingleCsv(df: DataFrame, fileName: String): Unit = {
  val target = s"$outputDir/$fileName"
  val tmpDir = s"$outputDir/${fileName}_tmp"
  df.coalesce(1).write.option("header", "true").mode("overwrite").csv(tmpDir)
  val partFile = dbutils.fs.ls(tmpDir).filter(_.name.endsWith(".csv")).head.path
  dbutils.fs.rm(target)
  dbutils.fs.cp(partFile, target)
  dbutils.fs.rm(tmpDir, recurse = true)
  println(s"CSV written to: $target")
}

// WITH tags - keep only rows where is_APA_Improvement_exist is false and the hookline does NOT
// cross a road element; CSV cannot serialize array<struct>, so emit the tag columns as JSON strings.
val finalScopedWithTags = finalScoped
  .filter(col("is_APA_Improvement_exist") === false
    && col("is_intersects_with_road_element") === false)
  .withColumn("old_tags", to_json(col("old_tags")))
  .withColumn("latest_tags", to_json(col("latest_tags")))

// WITHOUT tags - keep only rows where is_APA_Improvement_exist is false, the APT moved less than
// 30 meters, and the hookline does NOT cross a road element; drop both tag columns.
val finalScopedNoTags = finalScoped
  .filter(col("is_APA_Improvement_exist") === false
    && col("distance_in_meters") < 30
    && col("is_intersects_with_road_element") === false)
  .drop("old_tags", "latest_tags")

// DC relocation input format - derived from the no-tags scope: keep every existing column and
// append the columns the Data Correctness relocation process expects.
val finalRelocationScope = finalScopedNoTags
  // Product_orbis_id: orbisIdString with ':' replaced by '_' (14533:hi:lo -> 14533_hi_lo)
  .withColumn("Product_orbis_id", regexp_replace(col("orbisIdString"), ":", "_"))
  // Address component being corrected and its target (latest) location
  .withColumn("Product_Address_component", lit("metadata:apa:previous_location"))
  .withColumn("target_value", col("latest_wkt"))
  // Orbis_X / Orbis_Y parsed from old_wkt = POINT(X Y): X is the first coord (lon), Y the second (lat)
  .withColumn("Orbis_Y", regexp_extract(col("old_wkt"), """POINT\(([^ ]+) ([^)]+)\)""", 1))
  .withColumn("Orbis_X", regexp_extract(col("old_wkt"), """POINT\(([^ ]+) ([^)]+)\)""", 2))
  .withColumn("Relocation_Type", lit("BUILDING_FOOTPRINT"))

writeSingleCsv(finalScopedWithTags, "SEACO-6182-ESP-Relocation-Scope-with-tags-without-road-crossed.csv")
writeSingleCsv(finalScopedNoTags, "SEACO-6182-ESP-Relocation-Scope-without-road-crossed.csv")
writeSingleCsv(finalRelocationScope, "SEACO-6182-ESP-Relocation-DC-Input-format-without-road-crossed.csv")

println(s"Final scoped APT count: ${finalScoped.count()}")
println(s"finalScopedWithTags count: ${finalScopedWithTags.count()}")
println(s"finalScopedNoTags count: ${finalScopedNoTags.count()}")
println(s"finalRelocationScope count: ${finalRelocationScope.count()}")

In [ ]:
%scala
// Group the scoped APTs into distance_in_meters bands (mirrors the Excel IFS formula, with the
// 10-50 band split into 10-20/20-30/30-40/40-50 and the 100-150 band split into 100-125/125-150)
// and count each. "1000+ meters" is an extra catch-all for moves beyond the last band.

import org.apache.spark.sql.functions._

val distanceGrouped = finalScoped
  .withColumn("distance_group",
    when(col("distance_in_meters") <= 10, "0-10 meters")
      .when(col("distance_in_meters") <= 20, "10-20 meters")
      .when(col("distance_in_meters") <= 30, "20-30 meters")
      .when(col("distance_in_meters") <= 40, "30-40 meters")
      .when(col("distance_in_meters") <= 50, "40-50 meters")
      .when(col("distance_in_meters") <= 100, "50-100 meters")
      .when(col("distance_in_meters") <= 125, "100-125 meters")
      .when(col("distance_in_meters") <= 150, "125-150 meters")
      .when(col("distance_in_meters") <= 200, "150-200 meters")
      .when(col("distance_in_meters") <= 300, "200-300 meters")
      .when(col("distance_in_meters") <= 500, "300-500 meters")
      .when(col("distance_in_meters") <= 1000, "500-1000 meters")
      .otherwise("1000+ meters"))
  // group_order keeps the bands in ascending distance order for display
  .withColumn("group_order",
    when(col("distance_in_meters") <= 10, 1)
      .when(col("distance_in_meters") <= 20, 2)
      .when(col("distance_in_meters") <= 30, 3)
      .when(col("distance_in_meters") <= 40, 4)
      .when(col("distance_in_meters") <= 50, 5)
      .when(col("distance_in_meters") <= 100, 6)
      .when(col("distance_in_meters") <= 125, 7)
      .when(col("distance_in_meters") <= 150, 8)
      .when(col("distance_in_meters") <= 200, 9)
      .when(col("distance_in_meters") <= 300, 10)
      .when(col("distance_in_meters") <= 500, 11)
      .when(col("distance_in_meters") <= 1000, 12)
      .otherwise(13))
  .groupBy("group_order", "distance_group")
  .agg(count("*").as("apt_count"))
  .orderBy("group_order")
  .drop("group_order")

display(distanceGrouped)

In [ ]:
%scala
// Same distance bands as the previous cell, but additionally broken down by processingType.
// processingType is not a column on finalScoped; here it is taken from the latest_tags
// metadata:user value (null when the tag is absent). Counts are per (distance band, user).

import org.apache.spark.sql.functions._

val distanceProcessingTypeGrouped = finalScoped
  .withColumn("processingType",
    filter(col("latest_tags"), t => t.getField("tagKey").getField("key") === "metadata:user")
      .getItem(0).getField("value"))
  .withColumn("distance_group",
    when(col("distance_in_meters") <= 10, "0-10 meters")
      .when(col("distance_in_meters") <= 20, "10-20 meters")
      .when(col("distance_in_meters") <= 30, "20-30 meters")
      .when(col("distance_in_meters") <= 40, "30-40 meters")
      .when(col("distance_in_meters") <= 50, "40-50 meters")
      .when(col("distance_in_meters") <= 100, "50-100 meters")
      .when(col("distance_in_meters") <= 125, "100-125 meters")
      .when(col("distance_in_meters") <= 150, "125-150 meters")
      .when(col("distance_in_meters") <= 200, "150-200 meters")
      .when(col("distance_in_meters") <= 300, "200-300 meters")
      .when(col("distance_in_meters") <= 500, "300-500 meters")
      .when(col("distance_in_meters") <= 1000, "500-1000 meters")
      .otherwise("1000+ meters"))
  // group_order keeps the bands in ascending distance order for display
  .withColumn("group_order",
    when(col("distance_in_meters") <= 10, 1)
      .when(col("distance_in_meters") <= 20, 2)
      .when(col("distance_in_meters") <= 30, 3)
      .when(col("distance_in_meters") <= 40, 4)
      .when(col("distance_in_meters") <= 50, 5)
      .when(col("distance_in_meters") <= 100, 6)
      .when(col("distance_in_meters") <= 125, 7)
      .when(col("distance_in_meters") <= 150, 8)
      .when(col("distance_in_meters") <= 200, 9)
      .when(col("distance_in_meters") <= 300, 10)
      .when(col("distance_in_meters") <= 500, 11)
      .when(col("distance_in_meters") <= 1000, 12)
      .otherwise(13))
  .groupBy("group_order", "distance_group", "processingType")
  .agg(count("*").as("apt_count"))
  .orderBy("group_order", "processingType")
  .drop("group_order")

display(distanceProcessingTypeGrouped)

In [0]:
%scala
// Display the final scoped relocated-APT dataset (the regression candidates written to CSV above).
display(finalScoped)

In [0]:
%scala
// Display only the scoped records that moved more than 125 meters.
import org.apache.spark.sql.functions._

display(finalScoped.filter(col("distance_in_meters") > 125))

In [0]:
%scala
// Print the total number of scoped APTs that carry metadata:apa:improvement = yes.
// is_APA_Improvement_exist is already computed on finalScoped in Step 11 (expected to be 0 here,
// since such APTs are filtered out of the scope).
import org.apache.spark.sql.functions._

val apaImprovementCount = finalScoped.filter(col("is_APA_Improvement_exist")).count()
println(s"Scoped APTs with metadata:apa:improvement=yes (is_APA_Improvement_exist=true): $apaImprovementCount")

In [0]:
%scala
// Display the scoped dataset rows where is_APA_Improvement_exist is true.
import org.apache.spark.sql.functions._

display(finalScoped.filter(col("is_APA_Improvement_exist")))

In [ ]:
%scala
// Filter finalRelocationScope to a specific, provided set of orbisIdString values.
import org.apache.spark.sql.functions._

val targetOrbisIds = Seq(
  "14533:11516613567023742976:6462522384025228140",
  "14533:11516615605049950208:12337090556445643619",
  "14533:11516612161269989376:12408683332341800902",
  "14533:11516612091690156032:8657352373094294334",
  "14533:11516697644375539712:17463787382094542341",
  "14533:11353643755932024832:110968270774581162",
  "14533:11516693938624856064:8352561940289993810",
  "14533:11516632582436749312:14695717273606356948",
  "14533:11516713626122911744:10601425969594601349",
  "14533:11516712045738000384:8094718820014800394",
  "14533:11516712127610552320:15968809304189999873",
  "14533:11516602387719061504:16403166742172853185",
  "14533:11516805996335136768:18194713382448073926",
  "14533:11516616265396453376:10628644098842172973",
  "14533:11516616229759811584:11347411011164447280",
  "14533:11516713878862233600:1932340374624995164",
  "14533:11516695997559668736:15052256528166128848",
  "14533:11516613636048617472:2174496656555886325",
  "14533:11516617048416649216:4836721795925317199",
  "14533:11516614583492608000:6665094325229138753",
  "14533:11516614632243265536:9651045302071026599",
  "14533:11516614634143023104:4202222566554582830",
  "14533:11516614535979270144:8519692988276845236",
  "14533:11516614487359422464:1524160841043771057",
  "14533:11516614489018007552:12832912291791015408",
  "14533:11516614624426131456:10822290833367608970",
  "14533:11516614490227277824:10862008260236348237",
  "14533:11516614637545652224:6493305040467693434",
  "14533:11516708472797593600:2515923569783388861",
  "14533:11516708599995367424:5793121551147458379",
  "14533:11516614406632701952:8216713614477576664",
  "14533:11516617608363573248:3880363906947945380",
  "14533:11516617447355777024:4808609718038189731",
  "14533:11516617638840696832:13461598386299392628",
  "14533:11516617569633107968:7235414280526590446",
  "14533:11516617633685372928:7511285721049027180",
  "14533:11516617657628557312:5769667706739349615",
  "14533:11516711537096589312:14667023806531761242",
  "14533:11516617607564296192:17668462476077528024",
  "14533:11516617608133935104:8899857323850541345",
  "14533:11516614574745387008:8472750937924147498",
  "14533:11516708466566168576:2943916055094875982",
  "14533:11516614499688579072:622717501897125919",
  "14533:11516614489717145600:16684955205218808381",
  "14533:11516708463074148352:785387060593020424",
  "14533:11516614493183737856:3008901955934260580",
  "14533:11516614489805750272:1278400442168422697",
  "14533:11516614545491689472:7575886019616434080",
  "14533:11516708459355111424:14745865624604083150",
  "14533:14594451536934666240:1952486503886349535",
  "14533:11516614491650719744:16697140951230161760",
  "14533:11516708545637711872:1554889621940645028",
  "14533:11516708462908735488:6841724269252257418",
  "14533:11516614573105152000:5864444559578046104",
  "14533:11516708500018102272:6477203779537429299",
  "14533:11516614543782772736:4330674580022096535",
  "14533:11516614616696815616:12407889316974644153",
  "14533:11516614633464856576:288804241251585514",
  "14533:11516614476517933056:16795396410019956923",
  "14533:11516614638214643712:15908574692986902188",
  "14533:11516708622257422336:10145087559956034875",
  "14533:11516708464436510720:3217785526563985195",
  "14533:11516614491281620992:15392773482929278109",
  "14533:11516708452691935232:2403172077034760313",
  "14533:11516614622823120896:11595370718821339775",
  "14533:11516708569934528512:5763122358501974426",
  "14533:11516617643778703360:5560964765111488793",
  "14533:11516617628570157056:1233096068746999761",
  "14533:11516617666071953408:2166396200326573655",
  "14533:11516711425747779584:17126142712699013712",
  "14533:11516617601645346816:7743715288686662884",
  "14533:11516614614834282496:6486972556396487651",
  "14533:11516614588790276096:4719152743378155055",
  "14533:11516708454945587200:2511213101365670761",
  "14533:11516614521596739584:8796053378782053687",
  "14533:11516708470602924032:13376132631559694249",
  "14533:11516614542780858368:17573390710834672336",
  "14533:11516708448586760192:13098757834234281362",
  "14533:11516708449304510464:12469860094569630926",
  "14533:11516614477549207552:362382169184255614",
  "14533:11516708534875127808:10331487825951465571",
  "14533:11516708618023796736:403919349858896952",
  "14533:11516614530658795520:4857226097943373361",
  "14533:11516614536524791808:13356488896268705322",
  "14533:11516614594687729664:12834451017644897460",
  "14533:11516614473774596096:18063759665715870389",
  "14533:11516614543439626240:5740537737311520949",
  "14533:11516614608967499776:14710821389635639183",
  "14533:11516711530284253184:9527295157541815508",
  "14533:11516617649976311808:15217317550040599926",
  "14533:11516617636712087552:6339704259347415304",
  "14533:11516617606708658176:7809820923017116368",
  "14533:11516711554221146112:7556994595221429896",
  "14533:11516711472104275968:1002302748213092461",
  "14533:11516711472608903168:2324987954961201849",
  "14533:11516617650627477504:15116238928065524267",
  "14533:11516617656437637120:12981920826705285365",
  "14533:11516711551397330944:12483353622816806138",
  "14533:11516617634181349376:5614420513757643644",
  "14533:11516711471818276864:12837879557659652756",
  "14533:11516711431365525504:4936408129873240434",
  "14533:11516617505679933440:5857748323243462064",
  "14533:11516617568570114048:3947540237680477137",
  "14533:11516617434543489024:10188516895121187443",
  "14533:11516617619113312256:635443232972963380",
  "14533:11516617631278891008:4040393766264582878",
  "14533:11516617631236685824:16676232273935169506",
  "14533:11516711462748094464:6808010831132893298",
  "14533:11516613945520619520:9607668309341269085",
  "14533:11516614082501345280:10835013423958693668",
  "14533:11516614090600284160:1628306640062240557",
  "14533:11516614081548189696:2465372094716521642",
  "14533:11516614065125392384:1645468731695393062",
  "14533:11516617039477014528:13859249306877602375",
  "14533:11516615435616059392:8839416293673103024",
  "14533:11516616925391159296:6515730674973255868",
  "14533:11516711469704871936:14901456980754056281",
  "14533:11516617627325759488:9080990170585167174",
  "14533:11516617610696392704:5503031458278337175",
  "14533:11516711492062085120:11245033415903486990",
  "14533:11516617643302125568:15460492833530796581",
  "14533:11516617650561155072:9041909704008529091",
  "14533:11516711494206423040:5352117957320892630",
  "14533:11516709049554763776:7724399471484104146",
  "14533:11516708645961531392:10432851994290292160",
  "14533:11516617564761686016:11431648896255471607",
  "14533:11516617446900695040:16860946554823399221",
  "14533:11516617635324297216:12471125167489874512",
  "14533:11516711439915614208:13809821815510288556",
  "14533:11516711523598008320:12022485824202453461",
  "14533:11516617735150829568:4971462982300544608",
  "14533:11516711690091954176:5458760110461825793",
  "14533:11516617862849298432:4854701535882338753",
  "14533:11516711737940574208:12018481323075269173",
  "14533:11516711579844935680:7012150837092576640",
  "14533:11516711592043806720:481438407262467051",
  "14533:11516614067524796416:2645702799617771007",
  "14533:11516613902060552192:17735706754775431534",
  "14533:11516614439738867712:11002090304330363678",
  "14533:11516709084908552192:7463884331546080847",
  "14533:11516709150854545408:10773768611787973232",
  "14533:11516613890335637504:10290915685175171562",
  "14533:11516614439483277312:18443808690035248406",
  "14533:11516614376201453568:13406281649107696539",
  "14533:11516614083367206912:17408931890773983703",
  "14533:11516614467223355392:15325474611014560682",
  "14533:11516614469592875008:13502594535358667713",
  "14533:11516613856761806848:16377641620527982889",
  "14533:11516616967576682496:17732770697699563510",
  "14533:11516617046819930112:9686301712989286468",
  "14533:11516617052572942336:4618019333547446083",
  "14533:11516614082821160960:8029096745013283359",
  "14533:11516613865963847680:16335862905366111127",
  "14533:11516709156045258752:14829159526335735287",
  "14533:11516709149608050688:10891136529438129478",
  "14533:11516709150533681152:6448908011235838752",
  "14533:11516709032816345088:15054908587341678503",
  "14533:11516613610755391488:14027767079005920636",
  "14533:11516613633324154880:3528404434139285123",
  "14533:11516613801828483072:8706713900669855287",
  "14533:11516713626595557376:17716265540142130442",
  "14533:11516712003749871616:6205116114646122379",
  "14533:11516712127289163776:16211073154115721286",
  "14533:11516614377685188608:4485856152618474225",
  "14533:11516613948815769600:3299653736388463400",
  "14533:11516613893406130176:13590039641947604480",
  "14533:11516613943246520320:11428545090207925378",
  "14533:11516617043398164480:17884446220836833084",
  "14533:11516616946259132416:14864537719085414722",
  "14533:11516709088725106688:11493944219257080526",
  "14533:11516711964411232256:13980521997001165716",
  "14533:11516711709279584256:9832020124376908511",
  "14533:11516711813848039424:9065262562884420115",
  "14533:11516713708059951104:3470388357986322977",
  "14533:11516712127405031424:8958768280930251545",
  "14533:11516712129287487488:13651148117748400382",
  "14533:11516712129321828352:11504097826176548204",
  "14533:11516712127660359680:18075135246848724124",
  "14533:11516708959029100544:2419523255045651581",
  "14533:11516617867530403840:14858175213835042369",
  "14533:11516617864657829888:6355482496346845251",
  "14533:11516711691048255488:6043414371437504453",
  "14533:11516709136516055040:2682305731364169509",
  "14533:11516720649235922944:3895523714951352006",
  "14533:11516711857919950848:15521484509937918608",
  "14533:11516712388387733504:10489565685654671800",
  "14533:11516711719964049408:8618070546966059267",
  "14533:11516712095015567360:3337145033640820667",
  "14533:11516711991866097664:10583932792041543295",
  "14533:11516712129315012608:4597841479478835995",
  "14533:11516713737154527232:10709100059808683731",
  "14533:11516614058630250496:8307292344547122888",
  "14533:11516614052470652928:16253099766897153418",
  "14533:11516711699465437184:15344760278361381909",
  "14533:11516770552298930176:7071577460300882053",
  "14533:11516709074485182464:3638836932509188261",
  "14533:11516712232041119744:15961490146232334310",
  "14533:11516618801205542912:16590573451440863360",
  "14533:11516713737286385664:14887612921955515171",
  "14533:11516760426374823936:3502119691516341881",
  "14533:11516706913409630208:15124564727290799334",
  "14533:11516611889112875008:18043536810863771917",
  "14533:11516706645090828288:16997245952254574263",
  "14533:11516615403881693184:37170172489498223",
  "14533:11516578614579363840:1419691082686500211",
  "14533:11516578473782607872:6609298093404631427",
  "14533:11516616125144432640:12456652367488247132",
  "14533:11516615301973999616:1790249769936441702",
  "14533:11516712237423722496:1751627706323322632",
  "14533:11516602359997857792:4108349978147195313",
  "14533:11516612117199912960:7586840969951783066",
  "14533:11516728295317110784:9390749015484292699",
  "14533:11516616676039262208:3324494730618436110",
  "14533:11516713837739180032:14872420576440706628",
  "14533:11516598636674351104:553043806874983736",
  "14533:11516611755094114304:17018217769564279347",
  "14533:11516713110645571584:5522059549517757279",
  "14533:11516613608594276352:12637681509987131623",
  "14533:11516713104560160768:909383535714070007",
  "14533:11516611309085196288:1786600061754907284",
  "14533:11516724380688252928:5820324505299223935",
  "14533:11329506574399242240:11238768225053379427",
  "14533:11516710623489097728:10072203520692893008",
  "14533:11516720005735579648:13809806414215262486",
  "14533:11516712168134868992:14756332688272131439",
  "14533:11516720688629612544:9064675513619478996",
  "14533:11516711197939924992:12366916872165565470",
  "14533:11516605260023791616:4984287138122503904",
  "14533:11516612011041554432:13019520446924427078",
  "14533:11516711944001748992:13272536129569592960",
  "14533:11516696002376826880:10038279823854280204",
  "14533:11516710629667045376:4955457965382158593",
  "14533:11516704882088542208:3130285863428808711",
  "14533:11516713773446004736:15690665820744869298",
  "14533:11516617849779060736:1428741038190066376",
  "14533:11516711711740329984:11839276898859552071",
  "14533:11516617850152353792:1941430149538023301",
  "14533:11516658522840170496:10245130462515838917",
  "14533:11516603360076955648:16520901648998717524",
  "14533:11516603755847548928:6377919226775109385",
  "14533:11516713739754733568:1539601106210988238",
  "14533:11516712412074803200:12771309224988497995",
  "14533:11516615636420722688:16408711299624487347",
  "14533:11516720668119990272:11847641010495556928",
  "14533:11516616947568803840:7822679636301683795",
  "14533:11516713104452943872:3532010959289063310",
  "14533:11516574182066421760:17514139235686780886",
  "14533:11516611649709867008:5767575302067281647",
  "14533:11516618720287719424:5466417425519393774",
  "14533:11516617037435437056:1267789353862017338",
  "14533:11516616911890219008:11953375417652260756",
  "14533:11516578981039374336:5874970142531686455",
  "14533:11516713881291259904:8108729253902305815",
  "14533:11516612008574779392:7912359778219233053"
)

val finalRelocationScopeFiltered = finalRelocationScope
  .filter(col("orbisIdString").isin(targetOrbisIds: _*))

println(s"Matched ${finalRelocationScopeFiltered.count()} of ${targetOrbisIds.size} requested orbisIdString values")
display(finalRelocationScopeFiltered)